In [ ]:
from datasets import load_dataset
import pandas as pd

# 1. Download the Dataset
print("Downloading data...")
# We removed 'trust_remote_code' to avoid the warning
dataset = load_dataset("glopardo/sp500-earnings-transcripts", split="train")
df = pd.DataFrame(dataset)

# 2. Identify the correct column name
# The dataset authors named the column 'transcript', not 'text'.
# This logic automatically finds it.
if 'transcript' in df.columns:
    source_col = 'transcript'
elif 'contents' in df.columns:
    source_col = 'contents'
else:
    # Fallback: grab the longest string column (usually the text)
    print("Warning: Column name not found. Auto-detecting...")
    # Find column with longest average length
    source_col = df.astype(str).apply(lambda x: x.str.len()).mean().idxmax()

print(f"Found speech text in column: '{source_col}'")

# 3. Rename it to 'text' for consistency
# This is crucial so your later AI models know where to look.
df = df.rename(columns={source_col: 'text'})

# 4. Save the Data
# We save the last 100 rows to a CSV file.
df_small = df.tail(100).copy()
df_small.to_csv("project_data.csv", index=False)

print("\n------------------------------------------------")
print("SUCCESS! File 'project_data.csv' has been created.")
print("Sample Input (What your model will see):")
print(df_small.iloc[0]['text'][:200] + "...")
print("------------------------------------------------")

In [ ]:
import pandas as pd
from openai import OpenAI
import json
from tqdm import tqdm # Progress bar

# --- CONFIGURATION ---
API_KEY = "Ysk-proj-jaFF18YqKvBtkvc__LvzwwfEnKHG2_dcig5P5nq8HlnOR7mQZEs-lriMrsh-Lk69RZLx-WHR_kT3BlbkFJNGugZPNTrfoM1zYEe0vRvfCnKUQNjNzyjoelTKWn4nd4xfxgzyOmobDMAozwiYm8HnoNExyasA" 
client = OpenAI(api_key=API_KEY)

# Load your raw data
df = pd.read_csv("project_data.csv")

# We will store our training data here
training_data = []

# The "Teacher" System Prompt (The Secret Sauce)
# This forces the model to output the specific JSON structure we need.
system_prompt = """
You are a senior financial analyst. Your job is to extract 'Reasoning Alpha' from earnings calls.
You must output a JSON object with the following keys:
1. "summary_facts": Extract key numbers (Revenue, EPS, Guidance) mentioned.
2. "tone_analysis": Analyze the CEO's confidence (Cautious vs. Optimistic).
3. "reasoning_trace": A step-by-step logical proof of why this is Bullish or Bearish.
4. "signal": "Bullish", "Bearish", or "Neutral".
"""

def get_reasoning(transcript_text):
    # We only take the first 4000 characters to save money/tokens
    # (In a real job, you'd process the whole thing, but this is a prototype)
    truncated_text = transcript_text[:4000]

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini", # 'mini' is 10x cheaper and smart enough for this
            response_format={ "type": "json_object" }, # Force JSON output
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Analyze this transcript:\n\n{truncated_text}"}
            ],
            temperature=0.2 # Low temperature for more logical/consistent answers
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        return None

# --- MAIN LOOP ---
print("Starting the Teacher Phase...")

# Let's generate data for the first 20 rows first (Test Run)
# Change .head(20) to .head(100) when you are ready to spend ~$2.00
for index, row in tqdm(df.head(20).iterrows(), total=20):
    
    raw_text = row['text']
    reasoning_json = get_reasoning(raw_text)
    
    if reasoning_json:
        # Format for Llama-3 Fine-Tuning (Alpaca Format)
        entry = {
            "instruction": "Analyze the following earnings transcript and provide a structured trading signal with reasoning.",
            "input": raw_text[:4000], # The model input
            "output": reasoning_json  # The model target (The Reasoning)
        }
        training_data.append(entry)

# Save to a JSONL file (Standard format for training)
with open("fine_tuning_dataset.jsonl", "w") as f:
    for entry in training_data:
        json.dump(entry, f)
        f.write("\n")

print(f"Success! Generated {len(training_data)} training examples in 'fine_tuning_dataset.jsonl'.")